In [ ]:
%pip install -q "qualtran @ git+https://github.com/quantumlib/Qualtran.git@c58703d00b01c346d3e77d3913cf2d6e9e556be3"


In [ ]:
import hashlib
import json
import os
from pathlib import Path
import shutil
import sys
import traceback
import zipfile

DATASET = Path('/kaggle/input/datasets/evanpieser/qcompile-verifiable')
COMPETITION = Path('/kaggle/input/competitions/google-verifiable-quantum-circuit-compilation')
SAMPLE = COMPETITION / 'sample_submission'
WORK = Path('/kaggle/working')
CANDIDATE = WORK / 'prefix_j9'
RELEASE = WORK / 'submission_files'
REFERENCE_SCORE = 3355599588.3155365
sys.path.insert(0, str(COMPETITION / 'kickmix'))
sys.path.insert(0, str(DATASET / 'starter_kit'))
sys.path.insert(0, str(DATASET))
print('Using exact prefix candidate with verified fallback')


In [ ]:
from starter.task3_circuits.build_circuits import build_submission, format_summary
from starter.task1_factorization.solution_io import BASELINES_DIRECTORY
from verifier.reference import load_systems
from verifier.verify import verify

systems = load_systems()
factorization = BASELINES_DIRECTORY / systems['FeMoCo76o113e']['baseline']
candidate_summary = None
candidate_error = None
try:
    candidate_summary = build_submission(
        'FeMoCo76o113e', factorization,
        tree='log_depth', brot=12, batch_size=9, rounding='nearest',
        compiler='prefix', clean_cap=8,
        output_directory=CANDIDATE, overwrite=True,
    )
    print(format_summary(candidate_summary))
except Exception as exc:
    candidate_error = f'{type(exc).__name__}: {exc}'
    traceback.print_exc()

use_candidate = (
    candidate_summary is not None
    and float(candidate_summary['score']) < REFERENCE_SCORE
)
print('Use candidate:', use_candidate)


In [ ]:
if RELEASE.exists():
    shutil.rmtree(RELEASE)
release_dfthc = RELEASE / 'dfthcsos'
release_dfthc.mkdir(parents=True)
source = CANDIDATE if use_candidate else SAMPLE / 'dfthcsos'
for path in source.iterdir():
    if path.is_file() and (path.name == 'submission.npz' or path.name.startswith('angle_k_')):
        shutil.copy2(path, release_dfthc / path.name)
for name in ('dqi.kmb', 'invsqrt.kmb'):
    shutil.copy2(SAMPLE / name, RELEASE / name)

verification = verify('FeMoCo76o113e', release_dfthc)
assert verification['status'] == 'PASS'
assert verification['score'] <= REFERENCE_SCORE
print(json.dumps(verification, indent=2, default=float))


In [ ]:
def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

files = sorted(path for path in RELEASE.rglob('*') if path.is_file())
members = {str(path.relative_to(RELEASE)): sha256(path) for path in files}
expected = {'dqi.kmb', 'invsqrt.kmb', 'dfthcsos/submission.npz'}
expected |= {f'dfthcsos/angle_k_{k}.{ext}' for k in range(verification['num_circuits']) for ext in ('kmb',)}
if use_candidate:
    expected = {'dqi.kmb', 'invsqrt.kmb', 'dfthcsos/submission.npz'} | {
        f'dfthcsos/angle_k_{k}.kmx' for k in range(verification['num_circuits'])
    }
assert set(members) == expected, (set(members) - expected, expected - set(members))

archive = WORK / 'submission.zip'
temporary = WORK / 'submission.tmp.zip'
with zipfile.ZipFile(temporary, 'w', zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
    for path in files:
        zf.write(path, str(path.relative_to(RELEASE)))
with zipfile.ZipFile(temporary) as zf:
    assert zf.testzip() is None
    assert set(zf.namelist()) == expected
os.replace(temporary, archive)
report = {
    'status': 'PASS',
    'selected_source': 'prefix_j9' if use_candidate else 'official_baseline_fallback',
    'candidate_error': candidate_error,
    'candidate_score': None if candidate_summary is None else candidate_summary['score'],
    'reference_score': REFERENCE_SCORE,
    'score': verification['score'],
    'reference_over_cost': REFERENCE_SCORE / verification['score'],
    'projected_overall_score_not_official': (REFERENCE_SCORE / verification['score']) ** (1 / 3),
    'verification': verification,
    'archive_sha256': sha256(archive),
    'member_sha256': members,
}
(WORK / 'release_verification.json').write_text(json.dumps(report, indent=2, default=float) + '\n')
print('Release:', archive, archive.stat().st_size, 'bytes')
print('Selected:', report['selected_source'])
print('dfthcsos score:', verification['score'])
print('Projected overall (not official):', report['projected_overall_score_not_official'])
